In [ ]:
from datetime import date, datetime
from implied_volatility import IVAnalyzer
from models.black_scholes import BSAnalyzer

In [ ]:
def year_fraction(expiration_date: str, valuation_date: str | None = None, day_count: float = 365.0) -> float:
    expiration = datetime.strptime(expiration_date, "%Y-%m-%d").date()
    valuation = datetime.strptime(valuation_date, "%Y-%m-%d").date() if valuation_date else date.today()
    maturity = (expiration - valuation).days / day_count
    return max(maturity, 1e-8)


def prepare_contract(contract: dict) -> dict:
    normalized = dict(contract)
    if "time_to_maturity" not in normalized and "expiration_date" in normalized:
        normalized["time_to_maturity"] = year_fraction(
            expiration_date=normalized["expiration_date"],
            valuation_date=normalized.get("valuation_date"),
        )
    normalized.pop("expiration_date", None)
    normalized.pop("valuation_date", None)
    return normalized

In [ ]:
contract = {
    "spot": 253.87,
    "strike": 250,
    "expiration_date": "2027-03-19",
    "risk_free_rate": 0.0415,
    "market_price": 36.55,
    "option_type": "call",
}


In [ ]:
normalized_contract = prepare_contract(contract)

iv_result = IVAnalyzer().analyze(normalized_contract)
implied_vol = iv_result["implied_volatility"]
print(f"Implied Volatility: {implied_vol:.4%}")

pricing_contract = {k: v for k, v in normalized_contract.items() if k != "market_price"}
pricing_contract["volatility"] = implied_vol

analyzer = BSAnalyzer()

In [ ]:
result_bs = analyzer.analyze(pricing_contract)

In [ ]:
print("European:", result_bs)